<!-- dads-lab-header -->
<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Lab1_n8n_Workflow_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">⚡ M12 Lab 1 — Workflow Automation with n8n</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~45 min</p>
</div>

> **📌 Lab approach.** n8n runs as a Node.js server — it can't be installed inside Colab. This lab builds the **exact same mental model in Python**: triggers, processors, and actions wired into a DAG. Once you understand the paradigm here, you can deploy a real n8n instance on n8n.cloud in under 5 minutes and re-create every workflow from this lab using drag-and-drop nodes.


In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
import os
import importlib.util
if importlib.util.find_spec('dads5250') is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp, pretty_print, lab_pill, setup_openai,
    DEFAULT_CHAT_MODEL, DEFAULT_MINI_MODEL,
)

lab_pill('M12 Lab 1 — Workflow Automation with n8n')
client = setup_openai()


---
## 🔹 Part 1 — The n8n Mental Model

n8n (pronounced "n-eight-n", short for *nodemation*) is an open-source workflow automation platform. Every workflow is a **directed acyclic graph (DAG)** made of three kinds of nodes:

| Node type | Role | Examples |
|-----------|------|----------|
| **Trigger** | Starts the workflow | Gmail: New Email, Webhook, Schedule |
| **Processor** | Transforms or enriches data | LLM: classify, HTTP GET, If/Branch |
| **Action** | Ships the result | Send Email, Slack notify, Sheets append |

**Data flows** as a JSON payload. Each node **receives** the previous node's output, **transforms** it, and **passes** it forward. By the time the payload reaches the final action node it contains everything the action needs.

```
Trigger  →  Processor(s)  →  Action
  {}         {field: val}    {result: ...}
```

This lab builds a Python engine that executes exactly this pattern — then you use the same concepts on n8n.cloud.


In [ ]:
# ============================================================
# Part 1 — Python Workflow Engine
# A minimal DAG executor that mirrors n8n's execution model.
# ============================================================

import json
from typing import Callable, Any

class WorkflowNode:
    """A single node in the workflow DAG."""
    def __init__(self, name: str, kind: str, fn: Callable[[dict], dict]):
        self.name = name    # display label
        self.kind = kind    # 'trigger' | 'processor' | 'action'
        self.fn = fn        # payload_in -> payload_out

class Workflow:
    """Runs nodes left-to-right, accumulating payload state."""
    def __init__(self, name: str):
        self.name = name
        self.nodes: list[WorkflowNode] = []

    def add(self, node: WorkflowNode) -> 'Workflow':
        self.nodes.append(node)
        return self          # fluent API: workflow.add(a).add(b)

    def run(self, initial: dict = {}) -> dict:
        payload = dict(initial)
        print(f'\n▶ Running workflow: "{self.name}"')
        print(f'  Input: {json.dumps(payload, indent=2)}')
        for node in self.nodes:
            print(f'\n  [{node.kind.upper()}] {node.name}')
            payload = node.fn(payload)
            new_keys = {k: v for k, v in payload.items() if k.startswith('_') or True}
            print(f'  Payload: {json.dumps(payload, indent=2)}')
        print(f'\n✅ Workflow complete.')
        return payload

print('Workflow engine ready.')


---
## 🔹 Part 2 — Email Triage Bot

**Real-world scenario:** Your support inbox receives hundreds of emails daily. You want to:
1. Read the incoming email (trigger)
2. Classify urgency with an LLM (processor)
3. Route critical emails to Slack, log all to a sheet (actions)

In n8n this is three nodes. Below we build it as a Python workflow.


In [ ]:
# ============================================================
# Part 2 — Email Triage Bot
# ============================================================

# --- Node 1: Trigger — simulate a new inbound email ---
def gmail_trigger(payload: dict) -> dict:
    return {
        'from':    'alex@client.com',
        'subject': 'URGENT: Production is down',
        'body':    'Our checkout flow is returning 500 errors. Customers cannot pay. Please escalate NOW.',
    }

# --- Node 2: Processor — LLM classifies urgency ---
def llm_classify(payload: dict) -> dict:
    subject = payload.get('subject', '')
    body    = payload.get('body', '')
    resp = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        messages=[
            {'role': 'system', 'content': (
                'You are an email triage assistant. '
                'Classify the urgency of the email as exactly one of: CRITICAL, HIGH, NORMAL, LOW. '
                'Reply with JSON: {"urgency": "<level>", "reason": "<1 sentence>"}'
            )},
            {'role': 'user', 'content': f'Subject: {subject}\n\nBody: {body}'}
        ],
        response_format={'type': 'json_object'},
    )
    classification = json.loads(resp.choices[0].message.content)
    return {**payload, **classification}

# --- Node 3: Action — route based on urgency ---
def route_and_log(payload: dict) -> dict:
    urgency = payload.get('urgency', 'NORMAL')
    actions = []
    if urgency == 'CRITICAL':
        actions.append('Slack → #oncall: "CRITICAL email from " + payload["from"]')
        actions.append('PagerDuty alert triggered')
    actions.append(f'Sheets → Email_Log row appended (urgency={urgency})')
    return {**payload, 'actions_taken': actions}

# --- Wire the workflow ---
triage_bot = (
    Workflow('Email Triage Bot')
    .add(WorkflowNode('Gmail: New Email',     'trigger',   gmail_trigger))
    .add(WorkflowNode('LLM: classify urgency','processor', llm_classify))
    .add(WorkflowNode('Route + Log',          'action',    route_and_log))
)

result = triage_bot.run()


---
## 🔹 Part 3 — Feedback Digest Workflow

A second common pattern: **scheduled batch → LLM summarize → deliver digest**.

Every morning at 9 AM, this workflow:
1. Fetches the last 24 h of feedback rows from a database (trigger)
2. Summarizes trends with an LLM (processor)
3. Formats and emails a digest to the team (action)

Notice how the payload **grows** as each node adds its contribution.


In [ ]:
# ============================================================
# Part 3 — Feedback Digest Workflow
# ============================================================
import datetime

# Simulated feedback rows (in production this would be an HTTP GET to your DB)
FEEDBACK_ROWS = [
    {'score': 2, 'comment': 'Checkout was painfully slow.'},
    {'score': 4, 'comment': 'Great product, quick delivery.'},
    {'score': 1, 'comment': 'Support never replied — very frustrating.'},
    {'score': 5, 'comment': 'Best experience I have had with any SaaS tool.'},
    {'score': 3, 'comment': 'Works fine but the UI feels dated.'},
]

# --- Node 1: Schedule trigger ---
def schedule_trigger(payload: dict) -> dict:
    return {'trigger_ts': datetime.datetime.utcnow().isoformat(), 'rows_fetched': FEEDBACK_ROWS}

# --- Node 2: LLM summarizes trends ---
def llm_summarize(payload: dict) -> dict:
    rows = payload['rows_fetched']
    feedback_text = '\n'.join(f"Score {r['score']}: {r['comment']}" for r in rows)
    avg_score = sum(r['score'] for r in rows) / len(rows)

    resp = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        messages=[
            {'role': 'system', 'content': (
                'You are a customer-insight analyst. '
                'Given feedback rows, identify the top positive theme and the top pain point. '
                'Reply with JSON: {"top_positive": "...", "top_pain_point": "...", "headline": "one-line summary"}'
            )},
            {'role': 'user', 'content': feedback_text},
        ],
        response_format={'type': 'json_object'},
    )
    summary = json.loads(resp.choices[0].message.content)
    return {**payload, 'avg_score': round(avg_score, 2), **summary}

# --- Node 3: Format markdown body ---
def format_digest(payload: dict) -> dict:
    md = (
        f"## Daily Feedback Digest — {payload['trigger_ts'][:10]}\n"
        f"**Avg score:** {payload['avg_score']} / 5  |  **Rows:** {len(payload['rows_fetched'])}\n\n"
        f"**Headline:** {payload.get('headline', '')}\n\n"
        f"**Top positive:** {payload.get('top_positive', '')}\n"
        f"**Top pain point:** {payload.get('top_pain_point', '')}\n"
    )
    return {**payload, 'email_body_md': md}

# --- Node 4: Send email (simulated) ---
def send_email(payload: dict) -> dict:
    print('\n--- Email preview ---')
    print(payload['email_body_md'])
    return {**payload, 'email_sent': True, 'recipients': ['team@company.com']}

digest_wf = (
    Workflow('Daily Feedback Digest')
    .add(WorkflowNode('Schedule: 9 AM',        'trigger',   schedule_trigger))
    .add(WorkflowNode('LLM: summarize trends', 'processor', llm_summarize))
    .add(WorkflowNode('Format markdown',       'processor', format_digest))
    .add(WorkflowNode('Send Email',            'action',    send_email))
)

result = digest_wf.run()


---
## 🔹 Part 4 — From Python to n8n JSON

A real n8n workflow is stored as JSON. When you export a workflow from n8n you get a structure like the one below — each Python `WorkflowNode` maps directly to an n8n node object.

Understanding this format helps you:
- **Import/export** workflows between environments
- **Version-control** your automations in Git
- **Programmatically generate** n8n workflows from Python


In [ ]:
# ============================================================
# Part 4 — n8n JSON Export Format
# This is what n8n saves to disk / what you import at n8n.cloud
# ============================================================

def export_to_n8n_json(workflow: Workflow) -> dict:
    """Convert our Python Workflow to n8n-compatible JSON structure."""
    N8N_TYPE_MAP = {
        'trigger':   'n8n-nodes-base.trigger',
        'processor': 'n8n-nodes-base.function',
        'action':    'n8n-nodes-base.httpRequest',
    }
    nodes = []
    connections = {}
    for i, node in enumerate(workflow.nodes):
        nodes.append({
            'id':       f'node-{i}',
            'name':     node.name,
            'type':     N8N_TYPE_MAP.get(node.kind, 'n8n-nodes-base.function'),
            'position': [240 * i, 300],
            'parameters': {},
        })
        if i > 0:
            prev_name = workflow.nodes[i - 1].name
            connections.setdefault(prev_name, {'main': [[]]})['main'][0].append(
                {'node': node.name, 'type': 'main', 'index': 0}
            )
    return {
        'name':        workflow.name,
        'nodes':       nodes,
        'connections': connections,
        'active':      False,
        'settings':    {},
    }

n8n_export = export_to_n8n_json(triage_bot)
print(json.dumps(n8n_export, indent=2))


---
## 🔹 Part 5 — Branching with an If Node

Real workflows branch based on data. n8n's **If node** sends the payload down one of two paths depending on a condition. Below we implement the same pattern: an LLM classifier decides the branch.


In [ ]:
# ============================================================
# Part 5 — Conditional Branching
# ============================================================

class BranchNode:
    """Routes the payload to one of two sub-workflows based on a condition function."""
    def __init__(self, name: str, condition_fn, true_nodes: list, false_nodes: list):
        self.name = name
        self.condition_fn = condition_fn
        self.true_nodes  = true_nodes   # list of WorkflowNodes
        self.false_nodes = false_nodes

    def run(self, payload: dict) -> dict:
        branch_taken = self.condition_fn(payload)
        path = 'TRUE' if branch_taken else 'FALSE'
        nodes = self.true_nodes if branch_taken else self.false_nodes
        print(f'\n  [BRANCH] {self.name} → path: {path}')
        for node in nodes:
            print(f'    [{node.kind.upper()}] {node.name}')
            payload = node.fn(payload)
        return payload

# --- Condition: is the email CRITICAL? ---
def is_critical(payload: dict) -> bool:
    return payload.get('urgency') == 'CRITICAL'

# --- TRUE path: escalate ---
def slack_notify(p): return {**p, 'slack_alert': True,  'channel': '#oncall'}
def page_duty(p):    return {**p, 'pagerduty':   True}

# --- FALSE path: just log ---
def log_only(p): return {**p, 'logged': True, 'log_table': 'Email_Archive'}

# --- Rebuild triage bot with branching ---
def simulate_branch_run():
    payload = gmail_trigger({})
    print(f'\n▶ Running: "Email Triage with Branching"')
    print(f'  Trigger payload: {json.dumps(payload, indent=2)}')
    payload = llm_classify(payload)
    print(f'\n  [PROCESSOR] LLM: classify urgency')
    print(f'  Payload after classify: {json.dumps(payload, indent=2)}')
    branch = BranchNode(
        'If: urgency == CRITICAL',
        is_critical,
        true_nodes  = [WorkflowNode('Slack #oncall',   'action', slack_notify),
                       WorkflowNode('PagerDuty alert', 'action', page_duty)],
        false_nodes = [WorkflowNode('Log to archive',  'action', log_only)],
    )
    payload = branch.run(payload)
    print(f'\n✅ Final payload: {json.dumps(payload, indent=2)}')

simulate_branch_run()


---
## ✋ Hands-On — Build Your Own Workflow

**Task:** Wire a three-node workflow:
1. **Trigger** — returns a webhook payload with `event: "new_signup"` and `email: "user@example.com"`
2. **Processor** — calls the LLM to generate a short personalized welcome message (use `email` from payload)
3. **Action** — prints `"Email sent to <email>: <welcome_message>"`

Fill in the `-----` placeholders below.


In [ ]:
# ✋ Hands-On: Signup Welcome Workflow

# --- Node 1: Webhook trigger ---
def webhook_trigger(payload: dict) -> dict:
    return {
        'event': '-----',      # fill in: 'new_signup'
        'email': '-----',      # fill in: 'user@example.com'
    }

# --- Node 2: LLM generates welcome message ---
def llm_welcome(payload: dict) -> dict:
    email = payload.get('email', 'user')
    resp = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        messages=[
            {'role': 'system', 'content': '-----'},  # fill in: system prompt for a welcome email writer
            {'role': 'user',   'content': f'Write a 1-sentence welcome for: {email}'},
        ],
    )
    return {**payload, 'welcome_message': resp.choices[0].message.content.strip()}

# --- Node 3: Send welcome email (print) ---
def send_welcome(payload: dict) -> dict:
    print(f"Email sent to {payload.get('-----')}: {payload.get('-----')}")
    return {**payload, 'delivered': True}

# --- Wire and run ---
welcome_wf = (
    Workflow('Signup Welcome Bot')
    .add(WorkflowNode('Webhook',        'trigger',   webhook_trigger))
    .add(WorkflowNode('LLM: welcome',   'processor', llm_welcome))
    .add(WorkflowNode('Send Email',     'action',    send_welcome))
)

# welcome_wf.run()   # ← uncomment once you fill in the placeholders


---
## 🔬 Observational Exercise

Pick **one** of the workflows above (Email Triage or Feedback Digest) and trace it with **5 different inputs**.

For each input, record:
- The trigger payload
- The LLM classification / summary output
- Which action branch was taken

Where did the LLM help the most? Where did it just add latency compared to a simple `if/else` rule?


In [ ]:
# 🔬 Observational Exercise: Change the email body below and re-run the triage bot
SAMPLE_EMAILS = [
    ('Question about pricing', 'Hi, can you send me your enterprise pricing deck?'),
    ('Minor bug report',       'The tooltip on the settings page seems slightly off-center.'),
    ('URGENT: Data breach?',   'We may have detected unauthorized access to our account. Please call ASAP.'),
    ('Thank you!',             'Just wanted to say the product is fantastic. Really loving it.'),
    ('Invoice issue',          'The invoice for March is missing the PO number. Can you resend?'),
]

print('--- Triage Batch Run ---')
for subject, body in SAMPLE_EMAILS:
    payload = {'subject': subject, 'body': body}
    result  = llm_classify(payload)
    print(f'  [{result["urgency"]:8s}] {subject[:50]:50s}  → {result.get("reason", "")[:60]}')


---
## 🚀 Taking It to Real n8n (n8n.cloud)

Everything you built above maps 1:1 to a real n8n workflow. Here is the migration checklist:

| Python concept | n8n equivalent |
|---------------|----------------|
| `WorkflowNode('Gmail: New Email', 'trigger', ...)` | Gmail trigger node |
| `llm_classify(payload)` | OpenAI node (chat: message a model) |
| `json.loads(resp.choices[0].message.content)` | n8n JSON parse node |
| `BranchNode('If: urgency == CRITICAL', ...)` | n8n If node |
| `slack_notify(payload)` | Slack node (post message) |
| `{**payload, 'new_key': value}` | n8n automatically passes all fields forward |

**To deploy on n8n.cloud in 5 min:**
1. Sign up free at n8n.cloud
2. Create a new workflow
3. Add a **Manual Trigger** (for testing) or **Gmail** / **Webhook** trigger
4. Add an **OpenAI** node — set model, paste your system prompt
5. Add an **If** node — condition: `{{$json["urgency"] === "CRITICAL"}}`
6. Add **Slack** and **Google Sheets** action nodes
7. Click ▶ Execute to run


<hr>
<h2 style="color:#2e7d32;">🎉 Lab Complete</h2>

You built and ran three workflow patterns:

- ✅ **Email Triage Bot** — trigger → LLM classify → conditional action
- ✅ **Feedback Digest** — schedule → LLM summarize → format → send
- ✅ **Signup Welcome** — webhook → LLM personalize → deliver

The same DAG model powers n8n, Zapier AI, and every no-code automation platform. Once you know the mental model, the drag-and-drop UI is just a visual wrapper.
